# Thesis exploratory notebook — LOW MEMORY v3
GSE81608 and GSE86469. **Streams expression files one gene row at a time** rather than loading 40,000 × 1,600 into memory. Uses bounded expression sampling, extracts only four marker rows, validates every cell ID, and keeps datasets separate.

**Run each cell in order.** First upload the two expression CSVs (one at a time is fine), then the two metadata CSVs. Keep the runtime connected. Do not run `Run all` after uploading, because it will repeat upload prompts.

This notebook provides feasibility/QC only, not differential expression. RPKM and RSEM expected counts are different scales. Missing metadata are not inferred.


In [ ]:
from google.colab import files
from pathlib import Path
ROOT=Path('/content')
print('Upload expression CSV files (one or both). Repeat this cell for the second file if necessary.')
files.upload()
print('Expression files:',[p.name for p in ROOT.glob('*expression*.csv')])

In [ ]:
from google.colab import files
print('Upload metadata CSV files (one or both). Repeat this cell for the second file if necessary.')
files.upload()
print('Metadata files:',[p.name for p in ROOT.glob('*metadata*.csv')])

In [ ]:
import csv, re, gc, sys, json, zipfile, math
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import importlib.metadata as im
ROOT=Path('/content'); OUT=ROOT/'thesis_low_memory_results';OUT.mkdir(exist_ok=True)
EXPR={'GSE81608':'GSE81608_expression_RPKM.csv','GSE86469':'GSE86469_expression_RSEM_expected_counts.csv'}
META={'GSE81608':'GSE81608_GEO_sample_metadata.csv','GSE86469':'GSE86469_GEO_sample_metadata.csv'}
for f in list(EXPR.values())+list(META.values()):
    if not (ROOT/f).exists(): raise FileNotFoundError('Missing '+f+'; rerun the appropriate upload cell.')
print('All four input files found.')
print('Python:',sys.version.split()[0], 'pandas:',im.version('pandas'),'numpy:',im.version('numpy'))


## Metadata and expression header verification
We match **GSE81608** sample titles like `Pancreatic islet cell sample 1` to expression columns like `Sample_1`, and **GSE86469** original sample titles directly. This mapping is checked for uniqueness and complete agreement, never assumed to be valid without checking.

In [ ]:
ALIASES={
'donor_id':['donor_id','donor','individual_id'],
'disease_status':['disease_group','disease_status','disease','condition'],
'cell_type':['original_cell_type','cell_type','cell_type_annotation','celltype'],
'age':['age'],'sex':['sex','gender'],'bmi':['bmi'],'batch':['batch','batch_id'],
'geo_id':['geo_sample_id','geo_accession','sample_id','gsm'],
'title':['original_sample_title','sample_title','title']
}
def field(df,name,required=False):
    lookup={str(c).lower().strip():c for c in df.columns}
    for k in ALIASES[name]:
        if k in lookup:return lookup[k]
    if required:raise ValueError(f'{name} missing. Available: {list(df.columns)}')
    return None
def expression_id(ds,title):
    title=str(title).strip()
    if ds=='GSE81608':
        m=re.fullmatch(r'Pancreatic islet cell sample\s+(\d+)',title,re.I)
        return 'Sample_'+m.group(1) if m else title
    return title
data={}; audits=[]
for ds in EXPR:
    m=pd.read_csv(ROOT/META[ds],low_memory=False)
    with open(ROOT/EXPR[ds],newline='',encoding='utf-8-sig') as f:
        header=next(csv.reader(f))
    gene_header=header[0]; columns=[c.strip() for c in header[1:]]
    title_col=field(m,'title',True)
    m['cell_id']=m[title_col].map(lambda s:expression_id(ds,s))
    for key in ['donor_id','disease_status','cell_type','age','sex','bmi','batch']:
        src=field(m,key,required=key in ('donor_id','disease_status','cell_type'))
        m[key]=m[src] if src else pd.NA
    m['dataset']=ds
    match=(len(columns)==len(m) and len(columns)==len(set(columns)) and
           m.cell_id.is_unique and set(columns)==set(m.cell_id))
    audit=dict(dataset=ds,expression_columns=len(columns),metadata_rows=len(m),
               duplicate_expression_ids=len(columns)-len(set(columns)),
               duplicate_metadata_ids=int(m.cell_id.duplicated().sum()),
               unmatched_expression=len(set(columns)-set(m.cell_id)),
               unmatched_metadata=len(set(m.cell_id)-set(columns)),
               header_match_verified=bool(match),gene_identifier_field=gene_header)
    audits.append(audit)
    print(ds,audit)
    if not match:
        pd.DataFrame({'expression_only':pd.Series(sorted(set(columns)-set(m.cell_id))),
                      'metadata_only':pd.Series(sorted(set(m.cell_id)-set(columns)))}).to_csv(OUT/(ds+'_mismatch.csv'),index=False)
        continue
    m=m.set_index('cell_id').loc[columns].reset_index()
    data[ds]={'metadata':m,'columns':columns}
pd.DataFrame(audits).to_csv(OUT/'matching_audit.csv',index=False)
if len(data)!=2: raise ValueError('One or both datasets did not match; inspect matching_audit.csv and *_mismatch.csv before continuing.')


## Streaming expression QC
Only a fixed subset of up to 32 columns per gene is retained for distribution plots. **Missing, negative and zero counts are scanned over all expression values.** Marker rows are extracted using gene IDs; no arbitrary INS cutoff is used. Gene ID mapping is explicitly documented as requiring cross-check against the original study annotation.

In [ ]:
MARKERS={
'GSE81608':{'INS':'3630','GCG':'2641','SST':'6750','PPY':'5539'},
'GSE86469':{'INS':'ENSG00000254647','GCG':'ENSG00000115263','SST':'ENSG00000157005','PPY':'ENSG00000108849'}
}
qc=[]; marker_records=[]; dist={}
for ds in EXPR:
    print('Streaming',ds,flush=True)
    cols=data[ds]['columns']; m=data[ds]['metadata']
    chosen=np.linspace(0,len(cols)-1,min(32,len(cols)),dtype=int)
    sample=[]; gene_seen=set(); duplicate_genes=0; nrows=0
    missing=negative=zeros=total=0
    marker_found=set()
    with open(ROOT/EXPR[ds],newline='',encoding='utf-8-sig') as f:
        reader=csv.reader(f); next(reader)
        for row in reader:
            if not row:continue
            nrows+=1
            gene=row[0].strip().split('.')[0]
            if gene in gene_seen:duplicate_genes+=1
            else:gene_seen.add(gene)
            if len(row)!=len(cols)+1:
                raise ValueError(f'{ds}: row {nrows} has {len(row)} columns, expected {len(cols)+1}')
            # float conversion is confined to ONE gene row, then released
            vals=np.empty(len(cols),dtype=np.float32)
            for i,s in enumerate(row[1:]):
                try:vals[i]=float(s) if s.strip() else np.nan
                except (ValueError,OverflowError):vals[i]=np.nan
            finite=np.isfinite(vals)
            missing+=int((~finite).sum())
            negative+=int(np.sum(vals[finite]<0))
            zeros+=int(np.sum(vals[finite]==0))
            total+=len(vals)
            subset=vals[chosen]; subset=subset[np.isfinite(subset)&(subset>=0)]
            if len(subset):sample.extend(np.log1p(subset).tolist())
            for symbol,gid in MARKERS[ds].items():
                if gene==gid:
                    marker_found.add(symbol)
                    for cell,value,ct in zip(cols,vals,m['cell_type']):
                        marker_records.append({'dataset':ds,'cell_id':cell,'cell_type':ct,
                                               'marker':symbol,'gene_id':gid,'expression':float(value)})
            if nrows%10000==0:print(ds,nrows,'genes processed',flush=True)
    qc.append({'dataset':ds,'genes':nrows,'cells':len(cols),'missing_or_nonfinite':missing,
               'negative':negative,'zero_fraction':zeros/max(1,total-missing),
               'duplicate_gene_ids':duplicate_genes,'markers_found':','.join(sorted(marker_found))})
    dist[ds]=np.asarray(sample,dtype=np.float32)
    print('Completed',ds,qc[-1])
    del sample;gc.collect()
pd.DataFrame(qc).to_csv(OUT/'expression_qc_summary.csv',index=False)
pd.DataFrame(marker_records).to_csv(OUT/'marker_expression_long.csv',index=False)
print(pd.DataFrame(qc).to_string(index=False))

In [ ]:
manifest=pd.concat([data[d]['metadata'][['dataset','cell_id','donor_id','disease_status','cell_type','age','sex','bmi','batch']] for d in data],ignore_index=True)
manifest.to_csv(OUT/'cell_manifest.csv',index=False)
def is_beta(v):return bool(re.search(r'\bbeta\b',str(v),re.I))
manifest['is_beta']=manifest.cell_type.map(is_beta)
donors=manifest.groupby(['dataset','donor_id','disease_status'],dropna=False).agg(
 total_cells=('cell_id','size'),beta_cells=('is_beta','sum')).reset_index()
donors.to_csv(OUT/'donor_summary.csv',index=False)
print('Donor-level counts:');print(donors.to_string(index=False))
print('Low beta-cell donors (<=10):');print(donors[donors.beta_cells<=10].to_string(index=False))
print('Missing values:',manifest.isna().sum().to_dict())
conflicts=manifest.groupby(['dataset','donor_id']).disease_status.nunique(dropna=True)
print('Donors with conflicting disease labels:',int((conflicts>1).sum()))
fig,ax=plt.subplots(figsize=(max(10,len(donors)*0.45),5))
colors=['#e5a154' if str(s).strip().upper()=='T2D' else '#3b87b5' for s in donors.disease_status]
ax.bar(range(len(donors)),donors.beta_cells,color=colors)
ax.set_xticks(range(len(donors)),[f'{d}: {p}' for d,p in zip(donors.dataset.str[-3:],donors.donor_id)],rotation=90,fontsize=7)
ax.set(ylabel='Annotated beta cells',title='Beta cells per donor (orange=T2D; blue=other)')
fig.tight_layout();fig.savefig(OUT/'beta_cells_per_donor.png',dpi=140);plt.show();plt.close(fig)
for ds,values in dist.items():
    fig,ax=plt.subplots(figsize=(7,4))
    ax.hist(values,bins=65)
    ax.set(xlabel='log1p(expression)',ylabel='Sampled entries',title=f'{ds} expression distribution (sampled cells)')
    fig.tight_layout();fig.savefig(OUT/(ds+'_expression_distribution.png'),dpi=140);plt.show();plt.close(fig)


In [ ]:
marks=pd.DataFrame(marker_records)
if marks.empty:print('WARNING: no marker IDs found; check gene IDs and annotations.')
else:
    for ds,g in marks.groupby('dataset'):
        fig,axs=plt.subplots(1,4,figsize=(16,4))
        for ax,symbol in zip(axs,['INS','GCG','SST','PPY']):
            sub=g[g.marker==symbol]
            if sub.empty:
                ax.set_title(symbol+' not found');continue
            med=sub.groupby('cell_type',dropna=False)['expression'].median().sort_values(ascending=False).head(12)
            ax.bar(range(len(med)),np.log1p(med.clip(lower=0).to_numpy()))
            ax.set_xticks(range(len(med)),med.index.astype(str),rotation=70,ha='right',fontsize=7)
            ax.set_title(symbol);ax.set_ylabel('log1p(median expression)')
        fig.suptitle(ds+' marker check against original annotations')
        fig.tight_layout();fig.savefig(OUT/(ds+'_marker_check.png'),dpi=140);plt.show();plt.close(fig)
print('Marker rows:',len(marks),'Expected if all four found:',4*sum(len(data[d]['columns']) for d in data))


In [ ]:
versions={'python':sys.version,'packages':{p:im.version(p) for p in ['numpy','pandas','matplotlib']}}
(OUT/'software_versions.json').write_text(json.dumps(versions,indent=2))
(OUT/'README.txt').write_text('Inputs: original GEO expression and metadata CSVs. Streaming QC processes one gene at a time to reduce RAM. Expression distributions use up to 32 evenly spaced cells per gene; missing/negative/zero checks use all values. Confirm marker gene IDs against original study annotation. GSE81608 RPKM and GSE86469 RSEM expected counts are not directly comparable. Do not infer missing donor metadata or exclude low beta-cell donors without documenting it.\n')
archive=ROOT/'Thesis_LOW_MEMORY_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file():z.write(p,p.name)
print('Created:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
from google.colab import files
files.download(str(archive))
